In [1]:
from langchain_openai import ChatOpenAI 
from langchain_core.messages import HumanMessage, BaseMessage
from dotenv import load_dotenv
from langgraph.graph import StateGraph, START, END 
from typing import TypedDict, Annotated 
from langgraph.graph.message import add_messages
from langgraph.checkpoint.memory import MemorySaver

In [2]:
load_dotenv()

True

In [3]:
model = ChatOpenAI()

In [4]:
# State
class ChatState(TypedDict):
    messages: Annotated[list[BaseMessage], add_messages]
    

In [5]:
# Function for Chat Node
def chat_node(state: ChatState) -> ChatState:
    # Take the user query from the state
    messages = state['messages']
    
    # send to LLM
    response = model.invoke(messages)
    
    # response store in state
    return {'messages': [response]}

In [6]:
# Checkpoint
memory_saver = MemorySaver()

# Graph
graph = StateGraph(ChatState)

# Node
graph.add_node("Chat_Node", chat_node)

# Edges
graph.add_edge(START, "Chat_Node")
graph.add_edge("Chat_Node", END)

# Compile 
chatbot = graph.compile(checkpointer=memory_saver)

In [7]:
thread_id = '1'

while True:
    user_input = input("You: ")
    print("User: ", user_input)
    
    if user_input.strip().lower() in ["exit", "quit", "bye"]:
        break
    else:
        config = {'configurable': {'thread_id': thread_id}}
        response = chatbot.invoke({'messages': [HumanMessage(content=user_input)]}, config= config)
        
        print("AI: ", response['messages'][-1].content)

User:  hi
AI:  Hello! How can I assist you today?
User:  exit


In [8]:
chatbot.get_state(config=config)

StateSnapshot(values={'messages': [HumanMessage(content='hi', additional_kwargs={}, response_metadata={}, id='e114b8dd-9534-45c1-a3b2-6c0602fbe09c'), AIMessage(content='Hello! How can I assist you today?', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 9, 'prompt_tokens': 8, 'total_tokens': 17, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-3.5-turbo-0125', 'system_fingerprint': None, 'id': 'chatcmpl-C2HofA9WFUzCV27jKwBqTVUhC7DAk', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='run--d68186e2-04cc-4fd8-908b-ec2f22df1c06-0', usage_metadata={'input_tokens': 8, 'output_tokens': 9, 'total_tokens': 17, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})]}, next=(), config={'configurable':